In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ==========================================
# 1. PERSIAPAN DATASET (CIFAR-10)
# ==========================================
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalisasi pixel (0-255 menjadi 0-1)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Kategori kelas CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = 10

# Convert label ke one-hot encoding
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# ==========================================
# 2. DATA AUGMENTATION PIPELINE
# ==========================================
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)
datagen.fit(x_train)

# ==========================================
# 3. BANGUN CNN DARI AWAL (FROM SCRATCH)
# ==========================================
def build_cnn_scratch():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(32, 32, 3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

print("--- Training CNN From Scratch ---")
cnn_scratch = build_cnn_scratch()
start_time = time.time()
history_scratch = cnn_scratch.fit(
    datagen.flow(x_train, y_train_cat, batch_size=64),
    epochs=15, # Sesuaikan epoch
    validation_data=(x_test, y_test_cat)
)
scratch_time = time.time() - start_time

# ==========================================
# 4. IMPLEMENTASI TRANSFER LEARNING (VGG16)
# ==========================================
# Feature Extraction
print("\n--- Training Transfer Learning (Feature Extraction) ---")
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False # Freeze base model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

tl_model = Model(inputs=base_model.input, outputs=predictions)
tl_model.compile(optimizer=Adam(learning_rate=0.001), 
                 loss='categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
history_tl_extract = tl_model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=64),
    epochs=10,
    validation_data=(x_test, y_test_cat)
)

# Fine-Tuning (Unfreeze layer terakhir)
print("\n--- Training Transfer Learning (Fine-Tuning) ---")
base_model.trainable = True
# Freeze semua layer kecuali 4 layer terakhir VGG16
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Recompile dengan learning rate sangat kecil untuk fine-tuning
tl_model.compile(optimizer=Adam(learning_rate=1e-5), 
                 loss='categorical_crossentropy', metrics=['accuracy'])

history_tl_fine = tl_model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=64),
    epochs=5,
    validation_data=(x_test, y_test_cat)
)
tl_time = time.time() - start_time

# ==========================================
# 5. EVALUASI KOMPREHENSIF
# ==========================================
def evaluate_model(model, history, name, train_time):
    print(f"\n--- Evaluasi {name} ---")
    print(f"Waktu Training: {train_time:.2f} detik")
    
    # Inferensi
    start_infer = time.time()
    y_pred = model.predict(x_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    infer_time = time.time() - start_infer
    print(f"Waktu Inferensi (10k gambar): {infer_time:.2f} detik")
    
    # Classification Report (Precision, Recall, F1)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=class_names))
    
    # Plot Learning Curve (Accuracy & Loss)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(f'{name} - Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{name} - Loss')
    plt.legend()
    plt.show()

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_classes)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Panggil fungsi evaluasi
evaluate_model(cnn_scratch, history_scratch, "CNN From Scratch", scratch_time)
# Untuk model TL, Anda bisa menggabungkan history atau mengevaluasi hasil akhir fine-tuning
evaluate_model(tl_model, history_tl_fine, "Transfer Learning (VGG16 + Fine-Tuning)", tl_time)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
from sklearn.manifold import TSNE
from tensorflow.keras.models import Model

# Ambil satu sampel gambar dari data test untuk divisualisasikan
sample_idx = 12  # Silakan ubah index ini untuk melihat gambar lain
img_sample = x_test[sample_idx]
img_label = y_test[sample_idx]
img_batch = np.expand_dims(img_sample, axis=0) # Ubah shape jadi (1, 32, 32, 3)

print(f"Label Asli Gambar: {class_names[img_label[0]]}")
plt.imshow(img_sample)
plt.title("Gambar Original")
plt.axis('off')
plt.show()

# ==========================================
# 5.1 VISUALISASI FEATURE MAPS (Aktivasi Layer)
# ==========================================
def visualize_feature_maps(model, img_tensor, layer_name=None, layer_index=0):
    print("\n--- Visualisasi Feature Maps ---")
    # Buat model baru yang outputnya adalah output dari layer konvolusi tertentu
    if layer_name:
        target_layer = model.get_layer(layer_name).output
    else:
        target_layer = model.layers[layer_index].output
        
    feature_extractor = Model(inputs=model.inputs, outputs=target_layer)
    features = feature_extractor.predict(img_tensor)
    
    # Plot beberapa channel (misal 16 channel pertama)
    plt.figure(figsize=(10, 10))
    for i in range(16):
        if i < features.shape[-1]: # Pastikan layer punya minimal 16 channel
            plt.subplot(4, 4, i+1)
            plt.imshow(features[0, :, :, i], cmap='viridis')
            plt.axis('off')
    plt.suptitle(f'Feature Maps dari Layer: {layer_name or model.layers[layer_index].name}')
    plt.show()

# Contoh penggunaan pada CNN Scratch (Layer konvolusi pertama, index 0)
visualize_feature_maps(cnn_scratch, img_batch, layer_index=0)


# ==========================================
# 5.2 VISUALISASI FILTER BOBOT (Weight Visualization)
# ==========================================
def visualize_filters(model, layer_index=0):
    print("\n--- Visualisasi Filter/Bobot ---")
    layer = model.layers[layer_index]
    # Cek apakah layer memiliki bobot (misal Conv2D)
    if not 'conv' in layer.name:
        print("Layer ini bukan convolutional layer.")
        return
        
    filters, biases = layer.get_weights()
    # Normalize nilai filter ke 0-1 untuk visualisasi
    f_min, f_max = filters.min(), filters.max()
    filters = (filters - f_min) / (f_max - f_min)
    
    n_filters = min(16, filters.shape[3]) # Tampilkan 16 filter pertama
    plt.figure(figsize=(10, 10))
    for i in range(n_filters):
        f = filters[:, :, :, i]
        plt.subplot(4, 4, i+1)
        # Jika filter berwarna (RGB), gunakan langsung. Jika tidak, gunakan channel 0
        if f.shape[2] == 3:
            plt.imshow(f)
        else:
            plt.imshow(f[:, :, 0], cmap='gray')
        plt.axis('off')
    plt.suptitle(f'Filter Bobot dari Layer: {layer.name}')
    plt.show()

visualize_filters(cnn_scratch, layer_index=0)


# ==========================================
# 5.3 GRAD-CAM (Gradient-weighted Class Activation Mapping)
# ==========================================
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = Model(
        model.inputs, 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # Hitung gradien dari kelas target terhadap output layer konvolusi terakhir
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Kalikan setiap channel di feature map dengan "pentingnya" channel tersebut
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Terapkan ReLU (hanya ambil nilai positif)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img, heatmap, alpha=0.4):
    # Resize heatmap agar ukurannya sama dengan gambar asli
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    # Konversi heatmap ke RGB
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) # Fix warna OpenCV
    
    # Konversi gambar asli kembali ke rentang 0-255
    original_img = np.uint8(255 * img)
    
    # Gabungkan gambar asli dengan heatmap
    superimposed_img = cv2.addWeighted(heatmap, alpha, original_img, 1-alpha, 0)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(original_img)
    plt.title("Original")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(heatmap)
    plt.title("Heatmap")
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM Overlay")
    plt.axis('off')
    plt.show()

print("\n--- Visualisasi Grad-CAM (Pada Model Transfer Learning) ---")
# PERHATIAN: Nama layer ini ('block5_conv3') spesifik untuk arsitektur VGG16. 
# Jika menggunakan model lain, cek nama layernya menggunakan base_model.summary()
heatmap = make_gradcam_heatmap(img_batch, tl_model, 'block5_conv3')
display_gradcam(img_sample, heatmap)


# ==========================================
# 5.4 t-SNE / PCA UNTUK VISUALISASI FEATURE EMBEDDING
# ==========================================
print("\n--- Visualisasi t-SNE (Feature Embeddings) ---")
# Kita ambil fitur dari layer Dense terakhir sebelum softmax pada model VGG
# Cari layer dense yang kita tambahkan saat transfer learning (sebelum Dropout/Output)
layer_name_for_embedding = tl_model.layers[-3].name 

embedding_model = Model(inputs=tl_model.inputs, outputs=tl_model.get_layer(layer_name_for_embedding).output)

# Gunakan subset kecil (misal 1000 gambar) agar proses komputasi t-SNE tidak terlalu lama
subset_size = 1000
x_subset = x_test[:subset_size]
y_subset = y_test[:subset_size].flatten()

# Dapatkan vektor fitur (embedding) dari model
print("Ekstraksi fitur dari model (mohon tunggu)...")
features_subset = embedding_model.predict(x_subset, verbose=0)

# Jalankan t-SNE untuk mengurangi dimensi (dari 256 dimensi ke 2 dimensi)
print("Menjalankan t-SNE (ini membutuhkan waktu beberapa detik)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(features_subset)

# Plot hasil t-SNE
plt.figure(figsize=(12, 8))
scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1], 
                      c=y_subset, cmap='tab10', alpha=0.7)

# Tambahkan legenda kelas
handles, _ = scatter.legend_elements()
plt.legend(handles, class_names, loc="best", title="Kelas")
plt.title("t-SNE Visualization dari Fitur VGG16 (Transfer Learning)")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.show()